# Pairwise label consistency

Minimal notebook for comparing two pairwise ranking exports with Cohen's kappa and quadratic weighted kappa.

For binary pairwise choices, QWK should match Cohen's kappa; MAE is the disagreement rate; AMAE is class-balanced MAE.

In [1]:
import json
from pathlib import Path

import pandas as pd

EXPORT_BASES = [
    Path("data/label-consistency-test/pairwise"),
    Path("notebooks/data/label-consistency-test/pairwise"),
    Path("data/label-consistency-test"),
    Path("notebooks/data/label-consistency-test"),
]

base = next((p for p in EXPORT_BASES if (p / "ranks_1").exists() and (p / "ranks_2").exists()), None)
if base is None:
    raise FileNotFoundError("Could not find pairwise consistency exports with ranks_1/ and ranks_2/ folders")


def clip_name(uri):
    return Path(uri.split("://")[-1]).name


def read_export(folder):
    rows = []
    for path in sorted((base / folder).glob("*")):
        row = json.loads(path.read_text())
        if not row.get("result"):
            continue
        task_data = row["task"].get("data", row["task"])
        left = clip_name(task_data["left"])
        right = clip_name(task_data["right"])
        pair = tuple(sorted((left, right)))
        picked = left if row["result"][0]["value"]["selected"] == "left" else right
        rows.append({"pair": pair, f"choice_{folder}": int(picked == pair[1])})
    return pd.DataFrame(rows)


matching = read_export("ranks_1").merge(read_export("ranks_2"), on="pair")
print(f"Loaded {len(matching)} overlapping pairs from {base}")
matching.head()

Loaded 50 overlapping pairs from data/label-consistency-test/pairwise


,pair,choice_ranks_1,choice_ranks_2
0,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
1,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
2,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1
3,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,0
4,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,1,1


In [2]:
from dlordinal.metrics import amae as amae_score
from sklearn.metrics import cohen_kappa_score, mean_absolute_error

binary = matching[["choice_ranks_1", "choice_ranks_2"]].dropna().astype(int)

pd.DataFrame(
    [
        {
            "n_pairs": len(binary),
            "exact_agreement": (binary["choice_ranks_1"] == binary["choice_ranks_2"]).mean(),
            "mae": mean_absolute_error(binary["choice_ranks_1"], binary["choice_ranks_2"]),
            "amae": amae_score(binary["choice_ranks_1"], binary["choice_ranks_2"]),
            "cohen_kappa": cohen_kappa_score(binary["choice_ranks_1"], binary["choice_ranks_2"]),
            "qwk": cohen_kappa_score(binary["choice_ranks_1"], binary["choice_ranks_2"], weights="quadratic"),
        }
    ]
)


,n_pairs,exact_agreement,mae,amae,cohen_kappa,qwk
0,50,0.9,0.1,0.090476,0.774775,0.774775
